In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#Sprint 1

In [ ]:
!pip -q install pandas pyarrow scikit-learn
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
# Ajusta o caminho conforme a tua pasta
parquet_path = "/content/drive/MyDrive/Planapp-main/data_sample/cision_news_rows.parquet"
sample = pd.read_parquet(parquet_path, engine="pyarrow")

In [ ]:
sample.head(-100)

In [ ]:
print(sample.shape)
print(sample.columns)
display(sample.head())
display(sample.dtypes)
print(sample.isna().sum().sort_values(ascending=False).head(10))

# Exemplo: ver duplicados
print("Duplicados por corpo de noticia:", sample.duplicated(subset=["noticia"]).sum())

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

In [ ]:
import re

# Escolhe a coluna base
col_texto = 'titulo'

# Normalização simples do texto: minúsculas, remover múltiplos espaços, remover pontuação leve
def normalizar(s):
    s = str(s).lower()
    s = re.sub(r"[\:\;\,\.\!\?\(\)\[\]\-\–\—\"\']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

sample[col_texto] = sample[col_texto].astype(str).fillna("").map(normalizar)

from nltk.corpus import stopwords
stopwords_pt = set(stopwords.words('portuguese'))

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

#Vetorização TF-IDF e cálculo da matriz de similaridade
vectorizer = TfidfVectorizer(stop_words=list(stopwords_pt), ngram_range=(1,2), max_features=20000, min_df=2)
X = vectorizer.fit_transform(sample[col_texto])

# Similaridade e novo limiar (mais baixo aumenta grupos)
S = cosine_similarity(X, dense_output=False)
threshold = 0.72

n = S.shape[0]
visitados = np.zeros(n, dtype=bool)
grupos = []
for i in range(n):
    if visitados[i]:
        continue
    similares = S[i].toarray().ravel() >= threshold
    grupo_idxs = np.where(similares)[0].tolist()
    visitados[grupo_idxs] = True
    grupos.append(grupo_idxs)


In [ ]:
# para contagens
from collections import Counter
sizes = Counter(len(g) for g in grupos)
print('Distribuição de tamanhos de grupo:', sizes)
print('Grupos totais:', len(grupos))
print('Grupos com 1 notícia:', sum(1 for g in grupos if len(g)==1))


In [ ]:
#criar o grupo id no dataframe
# Mapa notícia -> id do grupo (a partir da lista `grupos`)
grupo_id_map = {idx: gid for gid, g in enumerate(grupos) for idx in g}
sample["grupo_id"] = sample.index.map(grupo_id_map.get).astype("Int64")


In [ ]:
# Tabela com número de notícias por grupo
contagem_por_grupo = (
    sample.groupby("grupo_id")
          .size()
          .reset_index(name="num_noticias")
)

# Lista com os ids dos grupos que têm exatamente 32 notícias p.e.
grupos_32 = contagem_por_grupo.loc[
    contagem_por_grupo["num_noticias"] == 32, "grupo_id"
].tolist()

print("Grupos com x notícias:", grupos_32)
print("Total de grupos com x notícias:", len(grupos_32))


In [ ]:
# Garante que cada notícia tem o id do grupo
# Passo 1: Gerar o campo grupo_id para cada notícia
grupo_id_map = {idx: gid for gid, g in enumerate(grupos) for idx in g}
sample["grupo_id"] = sample.index.map(grupo_id_map.get).astype("Int64")  # aceita NA

sample["is_lider"] = False  # Inicializa a coluna is_lider

In [ ]:

gid = grupos_32[0] #0 é a primeria posicao, 1 a segunda e etc
exemplo_4 = sample.loc[sample["grupo_id"] == gid, ["grupo_id","is_lider","titulo","data_publicacao"]].head(50)
display(exemplo_4)



In [ ]:
#escolher líder por grupo (primeira publicada)
def lider_primeira_publicada_seguro(df, grupo, col_data='data_publicacao'):
    import pandas as pd
    if not isinstance(grupo, (list, tuple, np.ndarray)) or len(grupo) == 0:
        return None
    idx_validos = [int(i) for i in grupo if 0 <= int(i) < len(df)]
    if len(idx_validos) == 0:
        return None
    datas = pd.to_datetime(df.loc[idx_validos, col_data], errors='coerce')
    valid_mask = datas.notna()
    if not valid_mask.any():
        return idx_validos[0]
    min_idx = datas[valid_mask].idxmin()
    return int(min_idx)

col_data = "data_publicacao"
lideres = [lider_primeira_publicada_seguro(sample, g, col_data=col_data) for g in grupos]

sample["is_lider"] = False  # inicializa
lideres_validos = [i for i in lideres if i is not None and 0 <= int(i) < len(sample)]
sample.loc[lideres_validos, "is_lider"] = True

# Finalmente: vê as colunas relevantes
print(sample[["grupo_id", "is_lider", col_texto, col_data]].head(30))


In [ ]:
detalhes = sample[sample["grupo_id"] == 7556].head(50)
print(detalhes[["titulo", "data_publicacao"]])
#todas as noticias de cada grupo - neste caso o grupo (id) 7556 tem 32 noticias como vimos

In [ ]:
sample.head(40)

_____________________

_____________________